# S5.M2-1 真实隐语 SGB 复核 S5.M1 的代理结论

**目的**：S5.M1 用明文直方图 GBDT 代理筛选方法。本步用真实隐语 SGB 复核：
- 代理与 SGB 在同配置下是否一致（[DR-M5-012](../../../registry/decision_records/DR-M5-012.yaml) · D-1：差的绝对值 ≤ 0.005）；
- I2 + I3、M6、M1、M1x 的增益在 SGB 上是否成立。

**运行方式**：
- 隐语 secretflow-lite 1.13.0b0，debug 单进程两方模拟，OU；
- 每个（任务，方法）拟合一次（seed 11），配置沿用 S5.M1 在验证集上选定的配置，不再调参；
- M1 / M1x 由标签方把交叉拟合的 logit 作为每个样本的初始预测，推理时在标签方加回，协议不动；
- M6 为 6 个 SGB 的概率平均。

**数据**：公开数据，**开发证据**，不代表汇丰效果。逐行打分只在私有目录。

In [1]:
import hashlib
import json
import logging
import os
import pathlib
import subprocess
import sys
import tempfile

import numpy as np
import pandas as pd
import yaml

HASH_PREFIX_LEN = 12
logging.getLogger().setLevel(logging.ERROR)        # 隐语运行日志不进 notebook
REPO_ROOT = pathlib.Path.cwd()
while not (REPO_ROOT / "AGENTS.md").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))
CONFIG_PATH = REPO_ROOT / "modules/m5_modeling/configs/s5_m2_sgb_confirm.yaml"
m2 = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
BASE_PATH = REPO_ROOT / m2["base_config"]
cfg = yaml.safe_load(BASE_PATH.read_text(encoding="utf-8"))
COMPONENTS = [REPO_ROOT / "modules/m5_modeling/components" / n
              for n in ("method_screen.py", "method_screen_data.py", "method_screen_sgb.py")]


def sha(path, n=HASH_PREFIX_LEN):
    return hashlib.sha256(pathlib.Path(path).read_bytes()).hexdigest()[:n]


git_sha = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True, cwd=REPO_ROOT).strip()
dirty = subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True,
                       cwd=REPO_ROOT).stdout.strip() != ""
print("config:", CONFIG_PATH.relative_to(REPO_ROOT), "sha256:" + sha(CONFIG_PATH))
print("base config:", BASE_PATH.relative_to(REPO_ROOT), "sha256:" + sha(BASE_PATH))
print("决策记录:", m2["decision_record"], "| step:", m2["step_id"], "|", m2["purpose"])
print("seed:", m2["seed"], "| seeds:", m2["seeds"])
print("git:", git_sha + ("（工作区含本步未提交改动）" if dirty else ""))
for c in COMPONENTS:
    print("组件:", c.relative_to(REPO_ROOT), "sha256:" + sha(c))
PRIV = pathlib.Path(os.environ[cfg["private_root_env"]])
RUN = pathlib.Path(os.environ[cfg["run_dir_env"]])
SGB = pathlib.Path(os.environ[m2["sgb_dir_env"]])
print("运行时代码指纹（会话脚本，逐段记录）：")
print((SGB / "sgb_fingerprint.txt").read_text(encoding="utf-8"))

config: modules/m5_modeling/configs/s5_m2_sgb_confirm.yaml sha256:d35149bb6156
base config: modules/m5_modeling/configs/s5_m1_method_screen.yaml sha256:761bae66da90
决策记录: registry/decision_records/DR-M5-012.yaml | step: S5.M2 | 真实 SGB 复核 S5.M1 的代理结论——I2 + I3、M6、M1、M1x 在 SGB 上是否成立，代理与 SGB 的差是否在 0.005 以内
seed: 11 | seeds: [11, 22, 33, 44, 55, 66]
git: 067ebcc（工作区含本步未提交改动）
组件: modules/m5_modeling/components/method_screen.py sha256:05e19e025d0b
组件: modules/m5_modeling/components/method_screen_data.py sha256:e4bfdf6fe3d4
组件: modules/m5_modeling/components/method_screen_sgb.py sha256:2e54c618305f
运行时代码指纹（会话脚本，逐段记录）：


6fe0bf5b78add29df8764539bd1f50fa2082de0aad0beec2b1c9f50fa98a1472  sgb_confirm.py
56929574cc556d9d75659830a56c8aa634dd150fe9d8012c70ad4711ab02fc90  screen.py
2026-11-03T00:20:02+08:00
aa5cc92d339b82a7a5cc9188bb6acb5e0e4baaa31e9ac115c6debacd9d60bd55  sgb_confirm.py
2026-11-03T02:56:04+08:00



## 1. 运行来源与仓库组件复现

复核由会话脚本逐段运行，指纹见上一格；中途为留足会话时限改过两次队列，脚本另加了 M6 成员逐个存档。本步把它整理为仓库组件 `method_screen_sgb`。下一格用仓库组件在 UCI 上重跑 M1 与优化版两次真实 SGB 训练，与运行时存档的打分逐行比较：要求逐行差 ≤ 1e-6（打分以 float32 存档，末位约 6e-8），且测试 AUROC 完全相同。

In [2]:
from modules.m5_modeling.components import method_screen as MS
from modules.m5_modeling.components import method_screen_sgb as SG

PRED_MATCH = 1e-6       # 逐行打分允许的差：存档为 float32，末位（约 6e-8）不同视为相同
P = MS.Params(cfg)
R = {r["dataset"]: json.loads((RUN / (r["dataset"] + ".json")).read_text(encoding="utf-8")) for r in m2["runs"]}
tmp = pathlib.Path(tempfile.mkdtemp())
rows = []
for item in m2["repro_check"]:
    n = item["dataset"]
    sc = MS.Screen(n, cfg, PRIV, tmp / "o")
    for m in item["methods"]:
        info = SG.confirm(P, sc, R[n], m, tmp)
        a, b = np.load(SGB / ("%s__%s.npy" % (n, m))), np.load(tmp / ("%s__%s.npy" % (n, m)))
        ye = sc.v["final"]["ev_y"]
        rows.append({"数据集": n, "方法": m, "棵数": info["trees"],
                     "逐行打分最大差": float(np.max(np.abs(a.astype(float) - b.astype(float)))),
                     "不同的行数": int(np.sum(a != b)), "行数": len(a),
                     "AUROC 相同": MS.auc(ye, a.astype(float)) == MS.auc(ye, b.astype(float)),
                     "SGB AUROC": info["sgb_auroc"], "拟合秒": info["fit_seconds"]})
rep = pd.DataFrame(rows)
print("仓库组件重跑真实 SGB，与运行时打分逐行比较：")
shown = rep.round(cfg["round_auc"])
shown["逐行打分最大差"] = rep["逐行打分最大差"].map("{:.2g}".format)
print(shown.to_string(index=False))
assert (rep["逐行打分最大差"] <= PRED_MATCH).all() and rep["AUROC 相同"].all()

[2026-11-03 11:10:07.920] [info] [bigint_spi.cc:79] The default library used for BigInt operations is openssl


[2026-11-03 11:10:16.806] [info] [thread_pool.cc:30] Create a fixed thread pool with size 1


仓库组件重跑真实 SGB，与运行时打分逐行比较：
     数据集      方法  棵数 逐行打分最大差  不同的行数   行数  AUROC 相同  SGB AUROC   拟合秒
uci_bank    F_M1   2   6e-08   4436 8238      True     0.8098   8.6
uci_bank F_opt_p  34       0      0 8238      True     0.8054 142.9


## 2. 代理与 SGB 的一致性，以及 SGB 上的方法增益

- 区间只含测试集抽样（客户簇 bootstrap），不含训练随机性；
- S5.M1 的代理对比是 6 个 seed 的结果，作对照列。

In [3]:
AUROC_MATCH = 1e-4      # 打分以 float32 存档，与运行时 float64 打分算出的 AUROC 允许的差
OUT = REPO_ROOT / m2["results_dir"]
OUT.mkdir(parents=True, exist_ok=True)
pairs = [tuple(x) for x in m2["pairs"]]
tol = m2["proxy_tolerance"]
meth, con = [], []
for run in m2["runs"]:
    n = run["dataset"]
    sc = MS.Screen(n, cfg, PRIV, tmp / "o")
    done = json.loads((SGB / (n + ".json")).read_text(encoding="utf-8"))
    sgb = {m: np.load(SGB / ("%s__%s.npy" % (n, m))).astype(float) for m in run["methods"]}
    proxy = {m: SG.proxy_pred(P, sc, R[n], m) for m in run["methods"]}
    ev = SG.evaluate(P, sc, R[n], sgb, proxy, pairs)
    for m, v in ev["methods"].items():
        assert abs(v["sgb_auroc"] - done[m]["sgb_auroc"]) < AUROC_MATCH
        assert abs(v["proxy_auroc"] - done[m]["proxy_seed11_auroc"]) < AUROC_MATCH
        d = v["sgb_minus_proxy"]
        meth.append({"dataset": n, "method": m, "trees": done[m]["trees"], "depth": done[m]["depth"], "bins": done[m]["bins"],
                     "sgb_auroc": v["sgb_auroc"], "proxy_auroc": v["proxy_auroc"], "sgb_minus_proxy": d["diff"],
                     "ci_low": d["ci_low"], "ci_high": d["ci_high"], "within_tolerance": abs(d["diff"]) <= tol,
                     "fit_seconds": done[m]["fit_seconds"]})
    for label, c in ev["contrasts"].items():
        s, q = c["sgb"], c["proxy_6seed"]
        con.append({"dataset": n, "contrast": label, "sgb_diff": s["diff"], "sgb_ci_low": s["ci_low"],
                    "sgb_ci_high": s["ci_high"], "sgb_p": s["p"], "proxy_6seed_diff": q["diff"],
                    "proxy_6seed_ci_low": q["ci_low"], "proxy_6seed_ci_high": q["ci_high"]})
meth, con = pd.DataFrame(meth), pd.DataFrame(con)
meth.to_csv(OUT / "sgb_vs_proxy.csv", index=False, float_format="%.5f")
con.to_csv(OUT / "sgb_contrasts.csv", index=False, float_format="%.5f")
print("同配置下 SGB 与代理（seed 11）：")
print(meth.round(cfg["round_auc"]).to_string(index=False))
print("SGB 上的方法对比（单次拟合，区间只含测试集抽样）与 S5.M1 代理（6 seed）：")
print(con.round(cfg["round_auc"]).to_string(index=False))
print("未执行或中止：")
for x in m2["not_run"]:
    print("·", x["dataset"], x["methods"], "—", x["reason"])

同配置下 SGB 与代理（seed 11）：
         dataset  method  trees  depth  bins  sgb_auroc  proxy_auroc  sgb_minus_proxy  ci_low  ci_high  within_tolerance  fit_seconds
        uci_bank   F_def    100      5    10     0.7951       0.7891           0.0060 -0.0018   0.0144             False        648.3
        uci_bank F_opt_p     34      4    33     0.8054       0.8070          -0.0016 -0.0065   0.0031              True        169.2
        uci_bank    F_M1      2      5    33     0.8098       0.8090           0.0008 -0.0007   0.0024              True          8.9
orange_appetency   F_def    100      5    10     0.8116       0.8052           0.0064 -0.0113   0.0237             False       3307.9
orange_appetency F_opt_p     72      3    33     0.8476       0.8496          -0.0021 -0.0067   0.0024              True       1779.2
    santander_g3 F_opt_p     33      6    33     0.8754       0.8763          -0.0010 -0.0038   0.0018              True       1327.4
    santander_g3    F_M1     41      6 

## 3. 读法

**仓库组件复现**：优化版逐位相同；M1 有约一半的行在 float32 末位不同（最大差约 6e-8），测试 AUROC 完全相同。会话脚本与仓库组件的公式相同，末位差异的来源未定位；仓库组件自身两次重跑逐位相同（会话中另行核对，不在本 notebook）。

**代理与 SGB 是否一致**（[DR-M5-012](../../../registry/decision_records/DR-M5-012.yaml) · D-1：差的绝对值 ≤ 0.005）：
- 13 组里 9 组在容差内；超出的 4 组都是 SGB 比代理**高** 0.006 左右：UCI 与 Orange 偏好的原版参数（约 10 桶）、mock 重合 80 与 COIL2000 的优化版。
- 原版参数只有约 10 个桶，隐语分位数草图与 XGBoost 的切点在粗分桶时差得更多，是可能的原因（未单独验证）。
- 按 D-1，这 4 个任务的代理结论降级：UCI 与 Orange 偏好的「原版 → 优化版」幅度以本步 SGB 结果为准；mock 重合 80 与 COIL2000 上只有 SGB 实测过的对比可作结论，其余标为「待 SGB 确认」。
- M1、M1x、M6 三个新方法在 SGB 上与代理的差都在 0.003 以内。

**SGB 上的方法增益**（单次拟合；区间只含测试集抽样）：
- I2 + I3（优化版 − 原版）：UCI +0.010 [+0.001, +0.019]，Orange 偏好 +0.036 [+0.017, +0.057]。成立，但幅度小于代理（+0.018 / +0.044），因为 SGB 上原版参数本身比代理高。
- M6 bagging：FedAds +0.011 [+0.006, +0.017]，成立；COIL2000 +0.003 [−0.003, +0.009]，区间含 0（代理 +0.008 显著）。
- M1x：FedAds +0.013 [+0.005, +0.021]，成立。
- M1：Santander G3 +0.003 [−0.001, +0.007]、UCI +0.004 [−0.001, +0.010]，区间都含 0；与代理一致（不显著）。UCI 上联邦部分只训 2 棵、拟合 9 秒（优化版 34 棵、169 秒），精度不低于优化版。

**未执行**：Orange 流失（已有 INNOVATIONS 的 SGB 证据）、mock 重合 80 的 M6（约 4.5 小时，推迟），见上一格末尾。

**边界**：公开数据开发证据，不代表汇丰效果；单次拟合不含训练随机性；debug 模拟下打分直接取回，只用于离线评估。技术侧自评，未经法务复核（W-001）。